### Comparación de varias medias: ANOVA

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, datediff, max as spark_max, min as spark_min, count, sum as spark_sum, avg, desc, asc
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
spark = SparkSession.builder \
    .appName("LRFM_HYM") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

In [3]:
# 1. Cargar datos y crear vista temporal
df = spark.read.parquet('merge_pyspark')
df.createOrReplaceTempView("customer_transactions")

In [4]:
# Exploramos la estructura del dataset
print("=== ESTRUCTURA DEL DATASET ===")
df.printSchema()
print(f"\nTotal de registros: {df.count():,}")

=== ESTRUCTURA DEL DATASET ===
root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullab

In [5]:
#Head 5
print("\n=== HEAD 5 REGISTROS ===")
df.limit(5).toPandas()


=== HEAD 5 REGISTROS ===


,customer_id,article_id,Fecha,price,sales_channel_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,...,section_name,garment_group_no,garment_group_name,detail_desc,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00030bc026428965d1868e4f3536f131feeedf5ce1b003...,673677015,2019-11-28,0.025407,2,673677,Henry polo (1),252,Sweater,Garment Upper body,...,Womens Tailoring,1003,Knitwear,"Jumper in a soft, fine knit with a ribbed polo...",NaN,NaN,ACTIVE,NONE,57,2c29ae653a9282cce4151bd87643c907644e09541abc28...
1,00030bc026428965d1868e4f3536f131feeedf5ce1b003...,673677010,2019-11-28,0.025407,2,673677,Henry polo (1),252,Sweater,Garment Upper body,...,Womens Tailoring,1003,Knitwear,"Jumper in a soft, fine knit with a ribbed polo...",NaN,NaN,ACTIVE,NONE,57,2c29ae653a9282cce4151bd87643c907644e09541abc28...
2,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,683001006,2019-02-10,0.016932,2,683001,Pattern 7p Socks,302,Socks,Socks & Tights,...,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Fine-knit socks in a soft cotton blend.,NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...
3,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,568571002,2018-10-24,0.013542,2,568571,Spartacus cheeky hipster,59,Swimwear bottom,Swimwear,...,"Womens Swimwear, beachwear",1018,Swimwear,"Fully lined, textured bikini bottoms with a lo...",NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...
4,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,723469007,2019-02-10,0.025407,2,723469,Kelly Push (Melbourne) ctn 2p,306,Bra,Underwear,...,Womens Lingerie,1017,"Under-, Nightwear",Push-up bras in soft cotton jersey with lace. ...,NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...
